In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns',None)


# APL Logistics Delivery Performance Analysis

## Objective
This project analyzes shipment delivery performance to evaluate SLA compliance, identify operational inefficiencies, and assess financial impact of delivery delays.

## Business Problem
APL Logistics experiences high late delivery rates. The goal is to diagnose the root cause of delay risk and provide actionable operational insights.

## Dataset Overview
- Total Orders: 180,519
- Features: 40 operational and financial variables

## Data Ingestion & Validation

In [7]:
df = pd.read_csv("APL_Logistics.csv", encoding='latin1')
df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Fname,Customer Id,Customer Lname,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,Order Item Discount,Order Item Discount Rate,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Product Name,Product Price,Shipping Mode
0,DEBIT,6,4,159.69,472.45,Late delivery,1,9,Cardio Equipment,Brownsville,EE. UU.,Richard,1,Hernandez,Consumer,TX,6303 Heather Plaza,78521.0,3,Footwear,25.953648,-97.507683,Pacific Asia,Mumbai,India,1,27.50,0.06,99.99,0.34,5,499.95,472.45,159.69,South Asia,Maharashtra,COMPLETE,Nike Men's Free 5.0+ Running Shoe,99.99,Standard Class
1,DEBIT,4,4,48.71,167.96,Shipping on time,0,29,Shop By Sport,Littleton,EE. UU.,Mary,2,Barrett,Consumer,CO,9526 Noble Embers Ridge,80126.0,5,Golf,38.375595,-104.726021,LATAM,San Pedro Sula,Honduras,2,31.99,0.16,39.99,0.29,5,199.95,167.96,48.71,Central America,Cortés,ON_HOLD,Under Armour Girls' Toddler Spine Surge Runni,39.99,Standard Class
2,DEBIT,4,4,87.36,181.99,Shipping on time,0,48,Water Sports,Littleton,EE. UU.,Mary,2,Barrett,Consumer,CO,9526 Noble Embers Ridge,80126.0,7,Fan Shop,38.375595,-104.726021,LATAM,San Pedro Sula,Honduras,2,18.00,0.09,199.99,0.48,1,199.99,181.99,87.36,Central America,Cortés,ON_HOLD,Pelican Sunstream 100 Kayak,199.99,Standard Class
3,DEBIT,6,4,-41.89,175.99,Late delivery,1,48,Water Sports,Littleton,EE. UU.,Mary,2,Barrett,Consumer,CO,9526 Noble Embers Ridge,80126.0,7,Fan Shop,38.375595,-104.726021,USCA,New York City,Estados Unidos,2,24.00,0.12,199.99,-0.24,1,199.99,175.99,-41.89,East of USA,Nueva York,COMPLETE,Pelican Sunstream 100 Kayak,199.99,Standard Class
4,DEBIT,6,4,10.00,40.00,Late delivery,1,24,Women's Apparel,Littleton,EE. UU.,Mary,2,Barrett,Consumer,CO,9526 Noble Embers Ridge,80126.0,5,Golf,38.375595,-104.726021,USCA,New York City,Estados Unidos,2,10.00,0.20,50.00,0.25,1,50.00,40.00,10.00,East of USA,Nueva York,COMPLETE,Nike Men's Dri-FIT Victory Golf Polo,50.00,Standard Class


In [9]:
df.shape

(180519, 40)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 40 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  object 
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  object 
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  object 
 9   Customer City                  180519 non-null  object 
 10  Customer Country               180519 non-null  object 
 11  Customer Fname                 180519 non-null  object 
 12  Customer Id                   

##late_delivery_risk == (real_shipping_days > scheduled_shipping_days)
## Delivery Gap Engineering

Delivery Gap = Actual Shipping Days - Scheduled Shipping Days

- Gap > 0 → Delayed
- Gap = 0 → On-Time
- Gap < 0 → Early

In [12]:
df["delivery_gap"] = df["Days for shipping (real)"] - df["Days for shipment (scheduled)"]

In [13]:
df["delivery_gap"].describe()

count    180519.000000
mean          0.565807
std           1.490966
min          -2.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           4.000000
Name: delivery_gap, dtype: float64

In [15]:
df["calculated_risk"] = np.where(df["delivery_gap"] > 0, 1, 0)

mismatch = df[df["calculated_risk"] != df["Late_delivery_risk"]]

len(mismatch)

4423

In [16]:
mismatch["delivery_gap"].value_counts()


delivery_gap
1    2690
2    1167
4     286
3     280
Name: count, dtype: int64

In [17]:
mismatch[["delivery_gap", "Late_delivery_risk"]].head()
mismatch["delivery_gap"].value_counts()

delivery_gap
1    2690
2    1167
4     286
3     280
Name: count, dtype: int64

In [18]:
df["alt_risk"] = np.where(df["delivery_gap"] >= 2, 1, 0)

alt_mismatch = df[df["alt_risk"] != df["Late_delivery_risk"]]

len(alt_mismatch)

59690

In [19]:
mismatch["Late_delivery_risk"].value_counts()

Late_delivery_risk
0    4423
Name: count, dtype: int64

In [20]:
df["delivery_status_calc"] = np.where(
    df["delivery_gap"] > 0, "Delayed",
    np.where(df["delivery_gap"] == 0, "On-Time", "Early")
)

df["delivery_status_calc"].value_counts()

delivery_status_calc
Delayed    103400
Early       43366
On-Time     33753
Name: count, dtype: int64

In [21]:


(df["delivery_status_calc"].value_counts(normalize=True) * 100).round(2)


delivery_status_calc
Delayed    57.28
Early      24.02
On-Time    18.70
Name: proportion, dtype: float64

In [22]:
df[df["delivery_status_calc"] == "Delayed"]["delivery_gap"].value_counts()

delivery_gap
1    60647
2    28718
3     7052
4     6983
Name: count, dtype: int64

In [24]:
df[df["delivery_status_calc"] == "Delayed"]["delivery_gap"].value_counts()

delivery_gap
1    60647
2    28718
3     7052
4     6983
Name: count, dtype: int64

### Key Insight:
Premium shipping modes exhibit unrealistic SLA commitments, leading to systemic delay classification.

In [25]:
pd.crosstab(df["Shipping Mode"], df["delivery_status_calc"], normalize="index") * 100

delivery_status_calc,Delayed,Early,On-Time
Shipping Mode,,,
First Class,100.000000,0.000000,0.000000
Same Day,47.827873,0.000000,52.172127
Second Class,79.730804,0.000000,20.269196
Standard Class,39.768171,40.246121,19.985708


In [26]:
df["Shipping Mode"].value_counts()

Shipping Mode
Standard Class    107752
Second Class       35216
First Class        27814
Same Day            9737
Name: count, dtype: int64

In [27]:
df.groupby("delivery_status_calc")["Order Profit Per Order"].mean()

delivery_status_calc
Delayed    21.585751
Early      22.469515
On-Time    22.532021
Name: Order Profit Per Order, dtype: float64

In [28]:
df.groupby("delivery_status_calc")["Order Item Discount Rate"].mean()

delivery_status_calc
Delayed    0.101729
Early      0.101815
On-Time    0.101294
Name: Order Item Discount Rate, dtype: float64

In [29]:
df.groupby("delivery_status_calc")["Sales"].mean()

delivery_status_calc
Delayed    203.338344
Early      204.645323
On-Time    203.978919
Name: Sales, dtype: float64

In [30]:
region_delay = pd.crosstab(
    df["Order Region"],
    df["delivery_status_calc"],
    normalize="index"
) * 100

region_delay.sort_values(by="Delayed", ascending=False).head(10)

delivery_status_calc,Delayed,Early,On-Time
Order Region,,,
Central Africa,60.703637,21.407275,17.889088
Western Europe,58.515622,23.615773,17.868605
South Asia,58.504721,23.658000,17.837278
South of USA,58.096415,24.103832,17.799753
Southeast Asia,57.983017,24.122025,17.894958
East of USA,57.975416,23.528561,18.496023
West Asia,57.497088,22.982193,19.520719
East Africa,57.451404,22.786177,19.762419
Eastern Europe,57.448980,23.290816,19.260204


In [31]:
pd.crosstab(
    [df["Order Region"], df["Shipping Mode"]],
    df["delivery_status_calc"],
    normalize="index"
) * 100

delivery_status_calc              Delayed      Early    On-Time
Order Region   Shipping Mode                                   
Canada         First Class     100.000000   0.000000   0.000000
               Same Day         33.333333   0.000000  66.666667
               Second Class     77.108434   0.000000  22.891566
               Standard Class   30.381944  50.520833  19.097222
Caribbean      First Class     100.000000   0.000000   0.000000
...                                   ...        ...        ...
West of USA    Standard Class   38.709004  40.442866  20.848130
Western Europe First Class     100.000000   0.000000   0.000000
               Same Day         48.896321   0.000000  51.103679
               Second Class     81.132769   0.000000  18.867231
               Standard Class   40.393343  40.355522  19.251135

[92 rows x 3 columns]

### Key Insight:
Delay patterns are relatively uniform across regions, indicating systemic scheduling inefficiency rather than geographic bottlenecks.

In [32]:
df.groupby("Shipping Mode")[
    ["Days for shipment (scheduled)", "Days for shipping (real)"]
].mean()

,Days for shipment (scheduled),Days for shipping (real)
Shipping Mode,,
First Class,1.0,2.000000
Same Day,0.0,0.478279
Second Class,2.0,3.990828
Standard Class,4.0,3.995907


### Key Insight:
Delayed shipments show slightly lower profit per order; however, the impact is modest compared to structural SLA misalignment.

In [33]:
df.groupby("Shipping Mode")["delivery_gap"].mean()

Shipping Mode
First Class       1.000000
Same Day          0.478279
Second Class      1.990828
Standard Class   -0.004093
Name: delivery_gap, dtype: float64

In [34]:
df[df["delivery_status_calc"] == "Delayed"]["Shipping Mode"].value_counts()

Shipping Mode
Standard Class    42851
Second Class      28078
First Class       27814
Same Day           4657
Name: count, dtype: int64

## Executive Summary

1. Overall delay rate: 57%.
2. Most delays are 1 day in severity.
3. Premium shipping modes suffer from unrealistic scheduling.
4. Standard Class contributes highest delay volume due to scale.
5. Root cause is SLA misalignment rather than operational collapse.